# 🚑 ZENTRA — Fall Detection Training (Kaggle)

เทรน **โมเดลตรวจจับการล้ม** ตัวใหม่ มาแทน TFLite Transformer ที่รับมรดกมาจาก
[punpayut/Fall-Detection](https://github.com/punpayut/Fall-Detection) (MIT)

---

## 🔴 ทำไมต้องเทรนใหม่ (เหตุผลเชิงกลไก ไม่ใช่ "อยากเทรน")

โมเดลปัจจุบันถูกเทรนบน **30 เฟรมที่ ~30 fps = หน้าต่าง 1.0 วินาที**

แต่ ZENTRA ตั้ง `FALL_LOOP_FPS=10` → หน้าต่าง 30 เฟรมของเรากินเวลา **3.0 วินาที**

**เราป้อนการเคลื่อนไหวที่ช้ากว่าทุกอย่างที่มันเคยเห็น 3 เท่า**
การล้มจริงใช้เวลาราว 0.8 วินาที — ในหน้าต่างของ upstream มันเต็มทั้ง 30 ช่อง
แต่ในหน้าต่างของเรามันกินแค่ ~8 ช่อง ที่เหลืออีก 22 ช่องคือ "คนนอนอยู่เฉย ๆ"

**ไม่มีการปรับ threshold ใด ๆ แก้เรื่องนี้ได้** — ต้องเทรนใหม่ที่ cadence เดียวกับที่ deploy

---

## 🎯 หลักการของ notebook นี้

> **คงสัญญา feature เดิมทุกตัวอักษร** → เทรนเสร็จแล้ว **วางไฟล์ `.tflite` ทับ
> `backend/assets/models/fall_detection_transformer.tflite` แล้วจบ
> ไม่ต้องแก้โค้ด inference แม้แต่บรรทัดเดียว**

สิ่งที่ต้องเหมือนเดิมเป๊ะ ๆ (ถ้าพลาดข้อใดข้อหนึ่ง โมเดลจะเห็นข้อมูลคนละแบบตอน deploy):

| | ค่า |
|---|---|
| pose model | `yolo11n-pose` (ตัวเดียวกับที่ deploy) · `imgsz=960` · `conf=0.25` |
| keypoints | 17 จุด COCO แต่ **เรียงตามตัวอักษร** ไม่ใช่ลำดับ COCO ← กับดักอันดับ 1 |
| features | 51 = (x, y, conf) × 17 วางที่ `[3i, 3i+1, 3i+2]` |
| พิกัด | **0–1** (หารด้วย W และ H) ไม่ใช่พิกเซล |
| normalize | ลบจุดกึ่งกลางสะโพก แล้วหาร x,y (ไม่ใช่ conf) ด้วย \|mid_shoulder_y − mid_hip_y\| |
| ไม่มี pose | เวกเตอร์ศูนย์ทั้งแถว |
| input | `(1, 30, 51)` float32 → output เดียว |

---

## ⚙️ ตั้งค่า Kaggle ก่อนรัน

แถบขวา (Session options):
1. **Accelerator → `GPU T4 x2`**
2. **Internet → `On`**
3. **Add Input** → เพิ่ม dataset การล้ม (ดูเซลล์ถัดไป)

▶️ กด **"Save Version" → "Save & Run All (Commit)"** แล้วปิดเบราว์เซอร์ได้เลย
(บทเรียนจากรอบเทรน person: อย่ารันทีละเซลล์แล้วปิดแท็บ — งานหาย)


## 1) เช็ค GPU + ติดตั้ง


In [ ]:
import torch, sys
print('CUDA :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'ยังไม่เปิด GPU → แถบขวา Accelerator → GPU T4 x2'

!pip -q install ultralytics
import ultralytics, tensorflow as tf
print('ultralytics', ultralytics.__version__, '| tensorflow', tf.__version__)


## 2) ตั้งค่า — ตัวเลขพวกนี้คือหัวใจ อย่าเปลี่ยนโดยไม่รู้ว่าทำอะไรอยู่


In [ ]:
NB_VERSION = 'v3 (2026-07-14) — Le2i nested-dir + detached .txt fix'
print('=' * 60)
print('ZENTRA Fall Training —', NB_VERSION)
print('=' * 60)

from pathlib import Path

# ── ต้องตรงกับ backend/.env และ backend/config.py ──────────────────────────
TARGET_FPS   = 10       # = FALL_LOOP_FPS ที่ deploy จริง (หน้าต่าง 30 เฟรม = 3.0 วินาที)
INPUT_TIMESTEPS = 30    # โครงสร้าง .tflite — ห้ามเปลี่ยน (fall_detector.py assert ไว้)
NUM_FEATURES = 51       # 17 keypoints × (x, y, conf)
POSE_IMGSZ   = 960      # = _YoloPoseExtractor.imgsz
POSE_CONF    = 0.25     # = ค่าใน _YoloPoseExtractor.extract()
MIN_COVERAGE = 0.6      # = cfg.FALL_MIN_POSE_COVERAGE

# ── labeling ──────────────────────────────────────────────────────────────
# หน้าต่างเป็น "ล้ม" ถ้าจังหวะที่ร่างกายถึงพื้น (fall_end) อยู่ในช่วง
#   [ปลายหน้าต่าง - 3.0s , ปลายหน้าต่าง + POST_SEC]
# POST_SEC ทำให้โมเดลยังยิงตอน "เพิ่งล้มแล้วนอนอยู่" ด้วย ซึ่งจำเป็น เพราะระบบจริง
# ต้องการ FALL_CONFIRM_FRAMES ติดกันหลาย tick ถึงจะเตือน
POST_SEC = 2.0

WORK = Path('/kaggle/working')
FEAT = WORK / 'features'; FEAT.mkdir(parents=True, exist_ok=True)
print(f'หน้าต่าง {INPUT_TIMESTEPS} เฟรม @ {TARGET_FPS} fps = {INPUT_TIMESTEPS/TARGET_FPS:.1f} วินาที')


## 3) สัญญา feature — **คัดลอกจาก `backend/utils/fall_detector.py` แบบคำต่อคำ**

> ⚠️ **ห้ามเขียนใหม่ ห้าม "ทำให้สะอาดขึ้น"**
> ฟังก์ชัน `normalize_skeleton_frame` มี branch ที่ไม่ชัดเจนหลายอัน
> (ไม่มีสะโพก → คืนค่าดิบ · shoulder_y == hip_y → ไม่ scale)
> ถ้าเขียนใหม่แล้วต่างแม้แต่นิดเดียว โมเดลจะเห็นข้อมูลคนละแบบตอน deploy


In [ ]:
import numpy as np

# ── ต่อไปนี้คือของจริงจาก backend/utils/fall_detector.py ─────────────────────
KEYPOINT_NAMES = [
    "Nose", "Left Eye", "Right Eye", "Left Ear", "Right Ear",
    "Left Shoulder", "Right Shoulder", "Left Elbow", "Right Elbow",
    "Left Wrist", "Right Wrist", "Left Hip", "Right Hip",
    "Left Knee", "Right Knee", "Left Ankle", "Right Ankle",
]
SORTED_NAMES = sorted(KEYPOINT_NAMES)          # ← กับดัก: เรียงตามตัวอักษร!
KP_INDEX = {n: i for i, n in enumerate(SORTED_NAMES)}
MIN_KP_CONF = 0.3
_COCO_ORDER = KEYPOINT_NAMES                   # ลำดับที่ ultralytics คืนมา


def _cols(name):
    i = KP_INDEX[name]
    return 3 * i, 3 * i + 1, 3 * i + 2


def normalize_skeleton_frame(f, min_confidence=MIN_KP_CONF):
    out = np.copy(f)
    lsx, lsy, lsc = _cols("Left Shoulder");  rsx, rsy, rsc = _cols("Right Shoulder")
    lhx, lhy, lhc = _cols("Left Hip");       rhx, rhy, rhc = _cols("Right Hip")

    mid_sh_x = mid_sh_y = np.nan
    v_ls, v_rs = f[lsc] > min_confidence, f[rsc] > min_confidence
    if v_ls and v_rs:   mid_sh_x, mid_sh_y = (f[lsx] + f[rsx]) / 2, (f[lsy] + f[rsy]) / 2
    elif v_ls:          mid_sh_x, mid_sh_y = f[lsx], f[lsy]
    elif v_rs:          mid_sh_x, mid_sh_y = f[rsx], f[rsy]

    mid_hip_x = mid_hip_y = np.nan
    v_lh, v_rh = f[lhc] > min_confidence, f[rhc] > min_confidence
    if v_lh and v_rh:   mid_hip_x, mid_hip_y = (f[lhx] + f[rhx]) / 2, (f[lhy] + f[rhy]) / 2
    elif v_lh:          mid_hip_x, mid_hip_y = f[lhx], f[lhy]
    elif v_rh:          mid_hip_x, mid_hip_y = f[rhx], f[rhy]

    if np.isnan(mid_hip_x) or np.isnan(mid_hip_y):
        return f                                    # upstream คืนเวกเตอร์ดิบ

    ref_h = np.nan
    if not np.isnan(mid_sh_y) and not np.isnan(mid_hip_y):
        ref_h = np.abs(mid_sh_y - mid_hip_y)
    scale = not (np.isnan(ref_h) or ref_h < 1e-5)

    for n in SORTED_NAMES:
        xc, yc, _ = _cols(n)
        out[xc] -= mid_hip_x
        out[yc] -= mid_hip_y
        if scale:
            out[xc] /= ref_h
            out[yc] /= ref_h
    return out


def _has_pose(f):
    return bool(np.any(f))


def pose_is_normalizable(f, min_confidence=MIN_KP_CONF):
    _, _, lhc = _cols("Left Hip");      _, _, rhc = _cols("Right Hip")
    _, _, lsc = _cols("Left Shoulder"); _, _, rsc = _cols("Right Shoulder")
    hip = f[lhc] > min_confidence or f[rhc] > min_confidence
    sho = f[lsc] > min_confidence or f[rsc] > min_confidence
    return bool(hip and sho)


print('SORTED_NAMES[0:5] =', SORTED_NAMES[:5])
print('→ สังเกตว่า "Left Ankle" มาก่อน "Nose" — นี่คือลำดับที่โมเดลต้องเห็น')


## 4) ข้อมูล — บอกว่าคลิปไหนล้มตอนไหน

### ⚠️ ตั้งค่าให้ถูกช่อง (พลาดกันบ่อย)

```python
LE2I_ROOT    = Path('/kaggle/input/datasets')   # ← Le2i ใส่ตรงนี้
MY_NEGATIVES = None                             # ← ตรงนี้ไว้ใส่คลิปที่คุณถ่ายเอง
```

ถ้าเอา Le2i ไปใส่ `MY_NEGATIVES` ระบบจะมองว่า **ทุกคลิปคือ "ไม่มีการล้ม"**
แล้วจะขึ้นว่า `มีการล้ม 0` → เทรนไม่ได้

### Le2i มีทั้งคลิปล้มและไม่ล้มอยู่ในชุดเดียวกัน

ไฟล์ `.txt` ที่คู่กับแต่ละคลิปบอกว่า **ล้มที่เฟรมไหน** (`0 0` = คลิปนี้ไม่มีการล้ม)

โครงสร้างบน Kaggle มีโฟลเดอร์ครอบซ้อนกันหลายชั้น และไฟล์ `.txt` **ไม่ได้อยู่ข้างวิดีโอ**
โค้ดข้างล่างจึง:
- **ไล่ลงโฟลเดอร์ครอบอัตโนมัติ** จนเจอชั้นที่มีหลายฉาก
- **ค้น `.txt` ทั้งฉาก** ไม่ใช่แค่ข้าง ๆ วิดีโอ (จับคู่ด้วยชื่อไฟล์)
- คลิปที่ **ไม่มี `.txt` เลย → ทิ้ง** เพราะไม่รู้ว่าล้มวินาทีไหน จะเอามาเป็น negative
  ก็ไม่ได้ (มันอาจมีการล้มอยู่แต่ไม่มีใคร label) — การเดาตรงนี้คือการสร้าง label ผิด

### 🔴 สิ่งที่ Le2i ให้ไม่ได้

Le2i ถ่ายในบ้าน/ออฟฟิศ 320×240 มุมกล้องระดับสายตา **และไม่มีท่า "นั่งกับพื้น" เลย**
ซึ่งเป็นท่าที่ทำให้ระบบเตือนผิดมากที่สุด และเป็นข้อที่ Gate 8 บังคับ
→ คลิปที่คุณถ่ายเองยังจำเป็นอยู่ (ใส่ที่ `MY_NEGATIVES`)


## 4a) 🔍 หา path จริงของ dataset ก่อน (รันเซลล์นี้ก่อนเสมอ)

Kaggle ตั้งชื่อโฟลเดอร์ตาม slug ของ dataset ซึ่ง **เดาไม่ได้**
รันเซลล์นี้เพื่อดูว่า Add Input เข้ามาแล้วมันไปโผล่ที่ไหน แล้วค่อยเอา path ไปแก้ `SOURCES` ในเซลล์ถัดไป


In [ ]:
import os
from pathlib import Path

VIDEO_EXT = {'.mp4', '.avi', '.mov', '.mkv'}
root = Path('/kaggle/input')

if not root.exists() or not any(root.iterdir()):
    print('ยังไม่ได้ Add Input เลย — แถบขวา → Add Input → ค้นหา dataset การล้ม')
else:
    for ds in sorted(root.iterdir()):
        vids = [q for q in ds.rglob('*') if q.suffix.lower() in VIDEO_EXT]
        anns = list(ds.rglob('*.txt'))
        print()
        print(f'DATASET: {ds}')
        print(f'   วิดีโอ {len(vids)} ไฟล์  ·  .txt (annotation?) {len(anns)} ไฟล์')
        for v in vids[:3]:
            print(f'   ตัวอย่าง: {v.relative_to(ds)}')
        if not vids:
            print('   ⚠️ ไม่เจอวิดีโอ — dataset นี้อาจเป็นภาพนิ่ง/เซนเซอร์ ใช้กับ notebook นี้ไม่ได้')

print()
print('ก๊อป path ที่ขึ้นต้นด้วย /kaggle/input/... ไปใส่ใน SOURCES เซลล์ถัดไป')


In [ ]:
import cv2, re
from collections import defaultdict

# ── แก้ให้ตรงกับที่เซลล์ 4a พิมพ์ออกมา ──────────────────────────────────────
LE2I_ROOT    = Path('/kaggle/input/datasets')   # ← Le2i (คลิปล้ม + ไม่ล้ม)
MY_NEGATIVES = None                             # ← คลิป negative ที่ถ่ายเอง (ยังไม่มีก็ None)

VIDEO_EXT = {'.mp4', '.avi', '.mov', '.mkv'}

assert MY_NEGATIVES is None or Path(MY_NEGATIVES) != LE2I_ROOT, \
    'MY_NEGATIVES ห้ามชี้ที่เดียวกับ LE2I_ROOT — ไม่งั้นคลิปล้มจะถูกนับเป็น negative หมด'


def real_root(root: Path) -> Path:
    """ไล่ลงโฟลเดอร์ครอบที่มีลูกตัวเดียว จนเจอชั้นที่มีหลายฉาก

    Kaggle ห่อ dataset ไว้หลายชั้น เช่น
      /kaggle/input/datasets/tuyenldvn/falldataset-imvia/<ฉาก>/...
    ถ้าไม่ไล่ลงมา 'ฉาก' จะกลายเป็นชื่อ owner ซึ่งมีค่าเดียว → แบ่ง split ไม่ได้
    """
    cur = root
    for _ in range(6):
        subs = [d for d in cur.iterdir() if d.is_dir()]
        if len(subs) == 1:
            cur = subs[0]
        else:
            break
    return cur


def parse_le2i(root):
    """Le2i: .txt สองบรรทัดแรก = เฟรมเริ่ม / เฟรมจบ ของการล้ม (0 0 = ไม่มีการล้ม)

    .txt ไม่ได้อยู่ข้างวิดีโอเสมอไป → ทำดัชนี stem → path ต่อ 'ฉาก' แล้วจับคู่
    (ชื่อไฟล์ซ้ำกันข้ามฉาก เช่น 'video (1)' มีอยู่ทุกฉาก จึงห้ามค้นข้ามฉาก)

    group = ชื่อคลิป+ฉาก (ไม่ใช่ฉากเฉย ๆ) เพราะ Le2i มีแค่ ~6 ฉาก — แบ่ง split
    ด้วยกลุ่มน้อยขนาดนั้นจะได้ val/test ที่ว่างเปล่า การแบ่งตามคลิปก็กัน
    window leakage ได้ครบเหมือนกัน ส่วน 'scene' เก็บไว้กันทั้งฉากเป็น test
    """
    out = []
    if not root.exists():
        return out
    base = real_root(root)
    print(f'ราก dataset จริง: {base}')

    scenes = [d for d in sorted(base.iterdir()) if d.is_dir()]
    if not scenes:
        scenes = [base]

    for sc in scenes:
        vids = sorted(p for p in sc.rglob('*') if p.suffix.lower() in VIDEO_EXT)
        txts = defaultdict(list)
        for t in sc.rglob('*.txt'):
            txts[t.stem.strip().lower()].append(t)

        n_ann = n_fall = 0
        for vid in vids:
            cands = txts.get(vid.stem.strip().lower(), [])
            if not cands:
                continue                       # ไม่มี label = ไม่รู้ว่าล้มไหม = ใช้ไม่ได้
            n_ann += 1
            nums = [int(x) for x in re.findall(r'\d+', cands[0].read_text(errors='ignore')[:200])[:2]]
            falls = []
            if len(nums) == 2 and nums[1] > nums[0] > 0:
                cap = cv2.VideoCapture(str(vid))
                fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
                cap.release()
                if not (0 < fps <= 240):
                    fps = 25.0
                falls = [(nums[0] / fps, nums[1] / fps)]
                n_fall += 1
            out.append({'video': vid, 'falls': falls,
                        'group': f'{sc.name}/{vid.stem}', 'scene': sc.name})
        flag = '' if n_ann else '   ⚠️ ไม่มี .txt เลย → ทิ้งทั้งฉาก'
        print(f'   {sc.name:<20} วิดีโอ {len(vids):>3}  ·  มี label {n_ann:>3}  ·  ล้ม {n_fall:>3}{flag}')
    return out


def scan_negatives(root, tag='mine'):
    out = []
    if root is None or not Path(root).exists():
        return out
    for vid in sorted(p for p in Path(root).rglob('*') if p.suffix.lower() in VIDEO_EXT):
        out.append({'video': vid, 'falls': [], 'group': f'{tag}/{vid.stem}', 'scene': tag})
    return out


MANIFEST = parse_le2i(LE2I_ROOT) + scan_negatives(MY_NEGATIVES)

n_pos = sum(1 for m in MANIFEST if m['falls'])
print(f'\nใช้ได้ {len(MANIFEST)} คลิป  ·  มีการล้ม {n_pos}  ·  negative {len(MANIFEST) - n_pos}')

assert MANIFEST, 'MANIFEST ว่าง — LE2I_ROOT ผิด? เช็คกับ path ที่เซลล์ 4a พิมพ์'
assert n_pos >= 20, (
    f'คลิปล้มมีแค่ {n_pos} คลิป\n'
    '  • ถ้าได้ 0 → คุณอาจเอา Le2i ไปใส่ MY_NEGATIVES แทน LE2I_ROOT\n'
    '  • หรือไฟล์ .txt อ่านไม่ออก / ไม่มีในชุดนี้')
if len(MANIFEST) - n_pos < 20:
    print('\n⚠️ คลิป negative น้อย → คาดหวังได้เลยว่าโมเดลจะเตือนผิดบ่อย')
    print('   ถ่ายคลิปเอง (นั่งกับพื้น ก้ม หมอบ เดิน) แล้วใส่ MY_NEGATIVES')


## 5) สกัด pose → 51 features (ขั้นที่แพงที่สุด — ทำครั้งเดียว แล้ว cache)

รีแซมเปิลคลิปลงเหลือ **10 fps ก่อน** แล้วค่อยสกัด → หน้าต่าง 30 เฟรมกินเวลา 3.0 วินาที
**เท่ากับที่ inference เห็นจริง** นี่คือหัวใจของ notebook ทั้งใบ

เซฟเป็น `.npy` → เทรนซ้ำได้ในไม่กี่นาทีโดยไม่ต้องรัน pose ใหม่


In [ ]:
from ultralytics import YOLO
import time

pose = YOLO('yolo11n-pose.pt')      # ตัวเดียวกับที่ deploy


def extract_clip(video_path, target_fps=TARGET_FPS, batch=32):
    """คืน (feats (T,51) หลัง normalize แล้ว, ok (T,) bool, times (T,))

    เลือก 'คนที่กล่องใหญ่ที่สุด' ในแต่ละเฟรม — dataset การล้มสาธารณะเป็นคนเดียวทั้งหมด
    ตอน inference จริงระบบทำ per-track อยู่แล้ว และ normalize เป็น hip-centred
    + torso-scaled → เวกเตอร์ที่ได้เหมือนกันไม่ว่าจะมาจากใคร"""
    cap = cv2.VideoCapture(str(video_path))
    src = cap.get(cv2.CAP_PROP_FPS) or 25.0
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if not (0 < src <= 240):
        src = 25.0
    step = src / target_fps

    frames, times, k = [], [], 0
    while True:
        idx = int(round(k * step))
        if n and idx >= n:
            break
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, fr = cap.read()
        if not ok:
            break
        frames.append(fr); times.append(k / target_fps); k += 1
    cap.release()
    if not frames:
        return np.zeros((0, NUM_FEATURES), np.float32), np.zeros(0, bool), np.zeros(0)

    feats = np.zeros((len(frames), NUM_FEATURES), np.float32)
    okv   = np.zeros(len(frames), bool)
    for b in range(0, len(frames), batch):
        chunk = frames[b:b + batch]
        res = pose.predict(chunk, imgsz=POSE_IMGSZ, conf=POSE_CONF,
                           device=0, verbose=False)
        for j, r in enumerate(res):
            if r.keypoints is None or r.boxes is None or len(r.boxes) == 0:
                continue
            H, W = chunk[j].shape[:2]
            boxes = r.boxes.xyxy.cpu().numpy()
            areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
            i = int(np.argmax(areas))                      # คนที่เด่นที่สุด

            kxy = r.keypoints.xy.cpu().numpy().astype(np.float32)
            kxy[..., 0] /= max(1, W)                       # ← 0-1 เหมือน inference
            kxy[..., 1] /= max(1, H)
            kcf = (r.keypoints.conf.cpu().numpy()
                   if r.keypoints.conf is not None else np.ones(kxy.shape[:2], np.float32))

            f = np.zeros(NUM_FEATURES, np.float32)
            for t, nme in enumerate(_COCO_ORDER):          # COCO order → slot เรียงตัวอักษร
                xc, yc, cc = _cols(nme)
                f[xc], f[yc], f[cc] = kxy[i, t, 0], kxy[i, t, 1], kcf[i, t]

            if _has_pose(f) and pose_is_normalizable(f):
                feats[b + j] = normalize_skeleton_frame(f)
                okv[b + j]   = True
            # ไม่ผ่าน → ปล่อยเป็นศูนย์ทั้งแถว (เหมือน 'no pose' ของ upstream เป๊ะ)
    return feats, okv, np.array(times)


t0 = time.time()
CACHE = []
for i, m in enumerate(MANIFEST, 1):
    npz = FEAT / (m['video'].stem + '.npz')
    if not npz.exists():
        f, ok, ts = extract_clip(m['video'])
        np.savez_compressed(npz, feats=f, ok=ok, times=ts)
    d = np.load(npz)
    CACHE.append({**m, 'feats': d['feats'], 'ok': d['ok'], 'times': d['times']})
    if i % 20 == 0 or i == len(MANIFEST):
        print(f'  {i}/{len(MANIFEST)}  ({time.time()-t0:.0f}s)')

tot = sum(len(c['feats']) for c in CACHE)
hit = sum(int(c['ok'].sum()) for c in CACHE)
print(f'\nรวม {tot} เฟรม @ {TARGET_FPS}fps  ·  pose อ่านได้ {hit} ({hit/max(1,tot):.1%})')


## 6) ตัดหน้าต่าง + ติด label

**สองเรื่องที่ทำให้ notebook นี้ต่างจากการเทรนแบบมั่ว ๆ:**

1. **ทิ้งหน้าต่างที่ pose coverage < 0.6** — เพราะตอน inference
   [`fall_detector.py`](https://github.com/) บังคับ `p_fall = 0.0` เมื่อ coverage ต่ำกว่านี้
   → หน้าต่างพวกนั้น **โมเดลไม่มีวันได้เห็นตอนใช้งานจริง** เทรนไปก็เสียเปล่าและทำให้เพี้ยน

2. **หน้าต่างที่คาบเกี่ยวการล้มแต่ไม่มีจังหวะกระแทกพื้น → ทิ้ง (ignore)**
   ไม่ใช่ทั้ง positive และ negative — มันคือ label noise ที่จะทำให้โมเดลสับสน


In [ ]:
WIN_SEC = INPUT_TIMESTEPS / TARGET_FPS


def make_windows(clip):
    X, y, g = [], [], []
    F, OK, T = clip['feats'], clip['ok'], clip['times']
    falls = clip['falls']
    for e in range(INPUT_TIMESTEPS - 1, len(F)):
        s = e - INPUT_TIMESTEPS + 1
        cov = float(OK[s:e + 1].mean())
        if cov < MIN_COVERAGE:
            continue                       # inference จะไม่มีวันเห็นหน้าต่างแบบนี้
        t_end, t_start = T[e], T[s]

        lab, ignore = 0, False
        for fs, fe in falls:
            if t_start <= fe <= t_end + POST_SEC:
                lab = 1                    # จังหวะกระแทกพื้นอยู่ในหน้าต่าง
            elif not (t_end < fs or t_start > fe + POST_SEC):
                ignore = True              # คาบเกี่ยว แต่ไม่มีจังหวะกระแทก → กำกวม
        if lab == 0 and ignore:
            continue
        X.append(F[s:e + 1]); y.append(lab); g.append(clip['group'])
    return X, y, g


X, y, G = [], [], []
for c in CACHE:
    a, b, d = make_windows(c)
    X += a; y += b; G += d

X = np.asarray(X, np.float32)
y = np.asarray(y, np.int32)
G = np.asarray(G)

print(f'หน้าต่างทั้งหมด : {len(X)}   shape {X.shape}')
print(f'  ล้ม (1)      : {int(y.sum())}  ({y.mean():.2%})')
print(f'  ไม่ล้ม (0)   : {int((1-y).sum())}')
assert X.shape[1:] == (INPUT_TIMESTEPS, NUM_FEATURES), X.shape
assert y.sum() > 50, 'หน้าต่าง positive น้อยเกินไป — ตรวจ label เวลาการล้มใน MANIFEST'


## 7) แบ่ง train / val / test

🔴 **กับดักที่โปรเจคนี้เคยติดมาแล้วในชุด PPE (leakage 24.5%)**

หน้าต่างแบบ sliding ซ้อนทับกันเกือบทั้งหมด — สองหน้าต่างที่ต่างกันแค่ 1 เฟรม
คือ "ภาพเดียวกัน" ในทางปฏิบัติ ถ้าแบ่งแบบสุ่มรายหน้าต่าง มันจะไปอยู่คนละ split
แล้วคะแนน val จะสวยหลอก ๆ

**กลยุทธ์ที่ใช้ (เมื่อมีแค่ Le2i):**

| split | มาจากไหน | ตอบคำถามว่า |
|---|---|---|
| **test** | 🔒 **กันไว้ทั้งฉาก** (เช่น Office ทั้งฉาก) | ย้ายไป**ฉากที่ไม่เคยเห็น**แล้วยังทำงานไหม — นี่คือข้อสอบที่ใกล้ "ย้ายไปโรงงานจริง" ที่สุดเท่าที่ Le2i ให้ได้ |
| **train / val** | ฉากที่เหลือ แบ่ง**ตามคลิป** | เลือกโมเดล/threshold |

หน้าต่างทุกอันของคลิปเดียวกันอยู่ split เดียวกันเสมอ → ไม่มี window leakage


In [ ]:
import hashlib

# ── เลือกฉากที่จะกันไว้เป็น test (ต้องมีคลิปล้มพอสมควร) ────────────────────
fall_by_scene = {}
for m in MANIFEST:
    if m['falls']:
        fall_by_scene[m['scene']] = fall_by_scene.get(m['scene'], 0) + 1

HOLDOUT_SCENE = None
if len(fall_by_scene) >= 2:
    # เลือกฉากที่เล็กที่สุดที่ยังมีคลิปล้ม >= 10 → กันไว้เป็น test โดยไม่กินข้อมูลเทรนมากเกิน
    cand = sorted((n, s) for s, n in fall_by_scene.items() if n >= 10)
    if cand:
        HOLDOUT_SCENE = cand[0][1]

if HOLDOUT_SCENE:
    print(f'🔒 กันไว้เป็น test ทั้งฉาก: {HOLDOUT_SCENE} '
          f'({fall_by_scene[HOLDOUT_SCENE]} คลิปล้ม)')
else:
    print('⚠️ มีฉากเดียว (หรือฉากอื่นเล็กเกิน) → แบ่ง test ตามคลิปแทน')
    print('   ตัวเลข test จะดูดีเกินจริง เพราะเป็นฉากเดิมที่โมเดลเคยเห็น')


def split_of(clip, val_pct=0.18):
    if HOLDOUT_SCENE and clip['scene'] == HOLDOUT_SCENE:
        return 'test'
    h = int(hashlib.sha256(clip['group'].encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    if not HOLDOUT_SCENE and h < 0.15:
        return 'test'
    return 'val' if h < (0.15 if not HOLDOUT_SCENE else 0.0) + val_pct else 'train'


SPLIT_OF_GROUP = {m['group']: split_of(m) for m in MANIFEST}
split = np.array([SPLIT_OF_GROUP[g] for g in G])
idx = {s: np.where(split == s)[0] for s in ('train', 'val', 'test')}

print(f"\n{'split':<7}{'หน้าต่าง':>11}{'ล้ม':>8}{'%ล้ม':>9}{'คลิป':>8}")
for s in ('train', 'val', 'test'):
    i = idx[s]
    if not len(i):
        print(f'{s:<7}{"— ว่าง —":>11}')
        continue
    print(f'{s:<7}{len(i):>11}{int(y[i].sum()):>8}{y[i].mean():>9.2%}{len(set(G[i])):>8}')

for s in ('train', 'val', 'test'):
    assert len(idx[s]) > 0, f'split "{s}" ว่างเปล่า — ปรับ val_pct หรือ HOLDOUT_SCENE'
    assert y[idx[s]].sum() > 0, f'split "{s}" ไม่มีตัวอย่างการล้มเลย — วัดผลไม่ได้'

# พิสูจน์ว่าไม่มีคลิปไหนข้าม split (= ไม่มี window leakage)
print()
for a, b in (('train', 'val'), ('train', 'test'), ('val', 'test')):
    ov = set(G[idx[a]]) & set(G[idx[b]])
    print(f"  {'✓' if not ov else '✗'} {a}↔{b}: {len(ov)} คลิปซ้ำ")
    assert not ov


## 8) Augmentation — สามอย่างที่คุ้มค่าที่สุด

| aug | ทำไม |
|---|---|
| **สลับซ้าย-ขวา** | กลับเครื่องหมาย x (หลัง normalize แล้ว x คือระยะจากสะโพก) + สลับ slot ซ้าย↔ขวา → ได้ข้อมูลเพิ่มเท่าตัวฟรี ๆ |
| **สุ่มลบ keypoint** | จำลอง**การถูกบัง** ซึ่งเกิดตลอดเวลาในโรงงานจริง — และเป็นสิ่งที่ dataset สาธารณะ (ถ่ายในบ้านโล่ง ๆ) แทบไม่มีเลย |
| **noise + time jitter** | กันโมเดลจำ pattern ที่ละเอียดเกินจริง |

> เฟรมที่เป็นศูนย์ (ไม่มี pose) ต้อง**ยังเป็นศูนย์**หลัง augment — ไม่งั้นเราจะสอนโมเดล
> ว่า "ไม่มี pose" หน้าตาเป็น noise ซึ่งขัดกับสิ่งที่ inference ป้อนให้


In [ ]:
# คู่ซ้าย-ขวา ใน slot ที่เรียงตามตัวอักษรแล้ว
_PAIRS = [(KP_INDEX[a], KP_INDEX['Right' + a[4:]])
          for a in SORTED_NAMES if a.startswith('Left')]


def aug_batch(w):
    """w: (B, 30, 51) — augment ในที่ (คืนสำเนาใหม่)"""
    w = w.copy()
    B = len(w)
    zero = ~np.any(w, axis=2)                       # (B,30) เฟรมที่ไม่มี pose

    # 1) mirror
    m = np.random.rand(B) < 0.5
    if m.any():
        sub = w[m]
        sub[:, :, 0::3] *= -1.0                     # x → -x  (hip-centred แล้ว)
        for li, ri in _PAIRS:                       # สลับ slot ซ้าย ↔ ขวา
            for c in range(3):
                a, b = 3 * li + c, 3 * ri + c
                sub[:, :, [a, b]] = sub[:, :, [b, a]]
        w[m] = sub

    # 2) keypoint dropout (จำลองการถูกบัง)
    drop = np.random.rand(B, 17) < 0.10
    for k in range(17):
        sel = drop[:, k]
        if sel.any():
            w[np.ix_(sel, np.arange(w.shape[1]), [3*k, 3*k+1, 3*k+2])] = 0.0

    # 3) noise เฉพาะ x,y (ไม่แตะ conf)
    n = np.random.normal(0, 0.02, w[:, :, 0::3].shape).astype(np.float32)
    w[:, :, 0::3] += n
    w[:, :, 1::3] += np.random.normal(0, 0.02, w[:, :, 1::3].shape).astype(np.float32)

    # เฟรมที่ไม่มี pose ต้องกลับไปเป็นศูนย์สนิท (inference ป้อนศูนย์แท้ ๆ)
    w[zero] = 0.0
    return w


import tensorflow as tf
from tensorflow import keras


class Seq(keras.utils.Sequence):
    def __init__(self, X, y, bs=128, augment=False, **kw):
        super().__init__(**kw)
        self.X, self.y, self.bs, self.aug = X, y, bs, augment
        self.i = np.arange(len(X))

    def __len__(self):
        return int(np.ceil(len(self.X) / self.bs))

    def __getitem__(self, b):
        j = self.i[b * self.bs:(b + 1) * self.bs]
        xb = self.X[j]
        return (aug_batch(xb) if self.aug else xb), self.y[j]

    def on_epoch_end(self):
        if self.aug:
            np.random.shuffle(self.i)


print('คู่ซ้าย-ขวา:', len(_PAIRS), 'คู่')


## 9) โมเดล + เทรน

Transformer เล็ก ๆ input `(30, 51)` → sigmoid
ใช้ **class weight** เพราะการล้มเป็นเหตุการณ์หายาก (positive ~2-8% ของหน้าต่าง)

เลือกโมเดลที่ดีที่สุดด้วย **val AUC-PR** ไม่ใช่ accuracy — accuracy บนข้อมูลที่ imbalance
คือตัวเลขที่ไร้ความหมาย (ทายว่า "ไม่ล้ม" ทุกครั้งก็ได้ 95% แล้ว)


In [ ]:
from tensorflow.keras import layers


def build_model(T=INPUT_TIMESTEPS, F=NUM_FEATURES, d=64, heads=4, blocks=2, batch_size=None):
    inp = keras.Input(shape=(T, F), batch_size=batch_size)
    x = layers.Dense(d)(inp)
    # positional encoding แบบเรียนรู้ได้ — ลำดับเวลาคือสาระของการล้ม
    pos = layers.Embedding(T, d)(tf.range(T))
    x = x + pos
    for _ in range(blocks):
        a = layers.MultiHeadAttention(num_heads=heads, key_dim=d // heads)(x, x)
        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(a))
        f = layers.Dense(d * 2, activation='relu')(x)
        f = layers.Dense(d)(f)
        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(0.1)(f))
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return keras.Model(inp, out)


model = build_model()
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(curve='PR', name='aucpr'),
             keras.metrics.Recall(name='recall'),
             keras.metrics.Precision(name='precision')],
)
model.summary()

tr, va = idx['train'], idx['val']
pos, neg = int(y[tr].sum()), int((1 - y[tr]).sum())
cw = {0: 1.0, 1: neg / max(1, pos)}
print(f'\nclass weight: {cw}  (ล้ม {pos} : ไม่ล้ม {neg})')

CKPT = str(WORK / 'fall_best.keras')
hist = model.fit(
    Seq(X[tr], y[tr], bs=128, augment=True),
    validation_data=Seq(X[va], y[va], bs=256),
    epochs=60,
    class_weight=cw,
    callbacks=[
        keras.callbacks.ModelCheckpoint(CKPT, monitor='val_aucpr',
                                        mode='max', save_best_only=True),
        keras.callbacks.EarlyStopping(monitor='val_aucpr', mode='max',
                                      patience=12, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_aucpr', mode='max',
                                          factor=0.5, patience=5),
    ],
    verbose=2,
)


## 10) เลือก threshold จากกราฟ PR — ไม่ใช่ใช้ 0.90 ที่รับมรดกมา

Gate 8 บอกว่า **precision ≥ 0.95** (เตือนผิดน้อยที่สุด) แล้ว **recall ≥ 0.90**

เราจึงหา threshold ที่**ต่ำที่สุดที่ยังได้ precision ≥ 0.95** บน **val**
แล้วรายงาน recall ที่ได้ — จากนั้นค่อยแตะ **test ครั้งเดียว**


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

pv = model.predict(X[idx['val']], batch_size=256, verbose=0).ravel()
yv = y[idx['val']]
prec, rec, thr = precision_recall_curve(yv, pv)

TARGET_PRECISION = 0.95
cand = [(t, p, r) for p, r, t in zip(prec[:-1], rec[:-1], thr) if p >= TARGET_PRECISION]
if cand:
    BEST_THR, bp, br = max(cand, key=lambda c: c[2])     # precision ผ่านแล้ว เอา recall สูงสุด
else:
    BEST_THR = 0.5
    bp = br = float('nan')
    print('⚠️ ไม่มี threshold ไหนได้ precision ≥ 0.95 เลย — โมเดลยังไม่ดีพอ')

print(f'AP (val)      : {average_precision_score(yv, pv):.4f}')
print(f'threshold ที่เลือก : {BEST_THR:.3f}  → val precision {bp:.3f} · recall {br:.3f}')

# ── แตะ test ครั้งเดียว ตอนจบ ────────────────────────────────────────────
pt = model.predict(X[idx['test']], batch_size=256, verbose=0).ravel()
yt = y[idx['test']]
pred = (pt >= BEST_THR).astype(int)
tp = int(((pred == 1) & (yt == 1)).sum()); fp = int(((pred == 1) & (yt == 0)).sum())
fn = int(((pred == 0) & (yt == 1)).sum())
R = tp / max(1, tp + fn); P = tp / max(1, tp + fp)
print(f'\n===== TEST (แตะครั้งเดียว) =====')
print(f'  recall    {R:.4f}   (เป้า ≥ 0.90)')
print(f'  precision {P:.4f}   (เป้า ≥ 0.95)')
print(f'  AP        {average_precision_score(yt, pt):.4f}')

print('\n⚠️ ตัวเลขข้างบนเป็น "ระดับหน้าต่าง" ไม่ใช่ "ระดับเหตุการณ์"')
print('   ระบบจริงมี confirm window + cooldown + spatial dedupe ทับอีกชั้น')
print('   → เลขที่ใช้ตัดสิน Gate 8 จริง ๆ ต้องมาจาก scripts/fall_eval.py เท่านั้น')


## 11) Export TFLite — และ**พิสูจน์**ว่ามันเหมือน Keras จริง

`fall_detector.py` assert ไว้ว่า input ต้องเป็น `(1, 30, 51)` เป๊ะ ๆ:

```python
got = tuple(int(v) for v in self._in["shape"])
if got != (1, INPUT_TIMESTEPS, NUM_FEATURES):
    raise RuntimeError(...)
```

เซลล์นี้จะสร้างโมเดล batch=1 → แปลง → **รันเทียบกับ Keras บน input เดียวกัน**
ถ้าผลต่างกันเกิน 1e-4 แปลว่าแปลงเพี้ยน อย่าเอาไปใช้


In [ ]:
# โมเดล inference: batch fix ที่ 1 (เทรนใช้ batch แปรผัน แต่ TFLite ต้องนิ่ง)
infer = build_model(batch_size=1)
infer.set_weights(model.get_weights())

conv = tf.lite.TFLiteConverter.from_keras_model(infer)
conv.optimizations = []                                  # float32 ล้วน ไม่ quantize
tfl = conv.convert()

OUT_TFLITE = WORK / 'fall_detection_transformer.tflite'
OUT_TFLITE.write_bytes(tfl)

# ── ตรวจสัญญา ────────────────────────────────────────────────────────────
it = tf.lite.Interpreter(model_content=tfl)
it.allocate_tensors()
ind, outd = it.get_input_details()[0], it.get_output_details()[0]
shape = tuple(int(v) for v in ind['shape'])
print('input shape :', shape, '| dtype', ind['dtype'].__name__)
print('output shape:', tuple(int(v) for v in outd['shape']))
assert shape == (1, INPUT_TIMESTEPS, NUM_FEATURES), \
    f'❌ shape {shape} → fall_detector.py จะ raise RuntimeError ทันที'

# ── พิสูจน์ว่า TFLite ให้ผลเท่ากับ Keras ─────────────────────────────────
probe = X[idx['test']][:64] if len(idx['test']) else X[:64]
ker = model.predict(probe, batch_size=64, verbose=0).ravel()
lite = []
for s in probe:
    it.set_tensor(ind['index'], s[None].astype(np.float32))
    it.invoke()
    lite.append(float(it.get_tensor(outd['index'])[0][0]))
diff = np.abs(ker - np.array(lite)).max()
print(f'ผลต่างสูงสุด Keras vs TFLite: {diff:.2e}')
assert diff < 1e-4, '❌ แปลงเพี้ยน — อย่าเอาไป deploy'
print(f'\n✅ {OUT_TFLITE.name}  ({len(tfl)/1024:.0f} KB)')


## 12) เอากลับไปใช้ใน ZENTRA

ไปที่แท็บ **Output** (แถบขวา) → ดาวน์โหลด **`fall_detection_transformer.tflite`**

```bash
# วางทับไฟล์เดิม (สำรองของเก่าไว้ก่อน)
cp backend/assets/models/fall_detection_transformer.tflite \
   backend/assets/models/fall_detection_transformer_upstream.tflite.bak
cp ~/Downloads/fall_detection_transformer.tflite \
   backend/assets/models/fall_detection_transformer.tflite
```

**ไม่ต้องแก้โค้ดใด ๆ** — นี่คือผลตอบแทนของการคงสัญญา feature ไว้ทั้งใบ

### แล้วตั้ง threshold ที่ notebook หามาได้ ใน `backend/.env`:

```bash
FALL_PROB_THRESHOLD=<BEST_THR ที่พิมพ์ออกมาในเซลล์ 10>
```

---

## 🔴 ขั้นสุดท้าย — อย่าเพิ่งเชื่อตัวเลขใน notebook นี้

ตัวเลขข้างบนเป็น **ระดับหน้าต่าง** (window-level) แต่ระบบจริงมี
`FALL_CONFIRM_FRAMES` + `CooldownGate` + spatial dedupe ทับอยู่อีกชั้น
**สิ่งที่โรงงานเห็นคือ "การเตือน" ไม่ใช่ "ความน่าจะเป็นต่อหน้าต่าง"**

รันตัววัดจริงเทียบกับ baseline:

```bash
python scripts/fall_eval.py --clips backend/data/fall_clips --dump-frames
```

แล้วเทียบกับเลขที่วัดไว้ **ก่อน**เปลี่ยนโมเดล (ดู [docs/FALL_STAGE0.md](../docs/FALL_STAGE0.md))
ถ้ามันไม่ดีขึ้นบนคลิปจริงของคุณ **แปลว่ามันไม่ดีขึ้น** ไม่ว่า AP บน val จะสวยแค่ไหน

### Gate 8 ที่ต้องผ่านจริง
- recall ≥ 0.90 · precision ≥ 0.95 · latency ≤ 4 วินาที
- **นั่งกับพื้น 5 นาที = 0 false positive** ← ด่านที่ยากที่สุด และไม่มีใน dataset สาธารณะ
